EBU6505 Reasoning and Agents

Introduction to LLM-based Agents
===

**Dr Chao Shu (chao.shu@qmul.ac.uk)**

In [1]:
import re
import ast
import operator
import io
import traceback
from datetime import date
from contextlib import redirect_stdout, redirect_stderr
from typing import Any

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
import warnings
warnings.filterwarnings('ignore')

from utils import get_completion, print_in_box, LLMModels

from dotenv import load_dotenv
load_dotenv()

plt.rcParams.update({
    'figure.figsize': (8, 4),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

## Setup
---

> 💡 Note:
>
> To achieve a reliable results for the complex tasks in this notebook, a modern LLM with sufficient scale is required. It is recommended to use OpenAI models hosted in [Github Models](https://github.com/marketplace?type=models) for FREE using your student accounts.
>
> To use the models through the helper functions (based on LiteLLM), you need to set the `GITHUB_API_KEY` in your `.env` file.

In [2]:
# Set default model configuration
# Ollama (local)
# model_api_config = LLMModels.OLLAMA_QWEN_3_4B.value

# GitHub Models (requires GITHUB_TOKEN in .env)
model_api_config = LLMModels.GITHUB_GPT_41_MINI.value

# model_api_config.temperature = 0.0
# model_api_config.max_tokens = 512

print(f"Using model: {model_api_config.provider}/{model_api_config.name}")

Using model: github/gpt-4.1-mini


In [3]:
result = get_completion(
    user_prompt="Please response with 'Hello, Agent!'",
    model_api_config=model_api_config,
)
hello_text = result[0] if isinstance(result, tuple) else result
print(hello_text)

Hello, Agent!


## Introduction to LLM-based Agents
---

A **language model** on its own can only take an input text (or in general, tokens) and generate an output text (or in general, tokens) in a single forward pass. An **LLM-based agent** uses LLM to act as the *brain* of a system that can:

- **Plan** a sequence of steps towards a goal with reasoning
- **Use tools** to interact with the outside world (search engines, calculators, APIs, code interpreters, …)
- **Observe** the results and revise its plan accordingly
- **Iterate** until the task is complete


![LLM-based Agent Overview](imgs/L06_agent-overview.png)

Image source: ["LLM Powered Autonomous Agents"](https://lilianweng.github.io/posts/2023-06-23-agent/)

Three key components of agentic systems:

| Component | Role | Example |
|---|---|---|
| **Planning** | Break the task into sub-goals; reflect on past actions | Chain-of-Thought, ReAct, Reflexion |
| **Memory** | Persist and retrieve information across steps | In-context history (short-term), vector store (long-term) |
| **Tool Use** | Extend capabilities beyond what the model knows | Web search, calculator, code execution, APIs |

### The Agentic Loop

The distinguishing feature of an agent is the **loop**: rather than producing a single response and stopping, the agent repeatedly **perceives** its environment, **reasons** about the next action, **acts**, and **observes** the result, until it decides the task is done.

The Agentic Loop is not just a theoretical concept, it is the fundamental engine that powers agentic systems like [OpenClaw](https://openclaw.ai/), allowing the agent to continuously gather information, make decisions, and react to its environment to solve a complex, multi-step problem.

In this lesson, we focus on two key agentic design patterns: **ReAct** and **Feedback Loops**.

## ReAct — Reasoning and Acting
---

ReAct ([Yao et al., 2022](https://arxiv.org/abs/2210.03629)) is a framework that combines **Chain-of-Thought (CoT)** and **Tool Calling**, so that LLMs are able to interleave reasoning traces and tool actions in the their output. It enable a language-based reasoning LLM to interact with external systems and data, showing the attributes of an agents that interacts with its environment.

![ReAct Framework](imgs/L06_ReAct_Framework.png)

Image Source: [ReAct: Synergizing Reasoning and Acting in Language Models](https://research.google/blog/react-synergizing-reasoning-and-acting-in-language-models/)

Under ReAct, each step follows a fixed three-part pattern:

```text
THINK: [the model reasons about what to do next]
ACT: tool_name[argument]
OBSERVE: [result returned by the tool — injected by the orchestrator]
```
The model generates `THINK` and `ACT`; the **orchestrator** (implemented by agentic frameworks, such as LangGraph) executes the action and
appends the `OBSERVE` before the next turn. This continues until the model produces a **final answer**.

**Why does this work better than plain CoT?**
- No need to put all information and data into the prompt.
- Solve problems in an active and autonomous way through interaction with the environment.
- The model's reasoning is *grounded* at each step: it cannot hallucinate facts that a tool has already contradicted.
- Tool results act as a continuous reality check, preventing error accumulation across long reasoning chains.

### Define a Tiny Tool Environment

Let's define a few tools and data our LLMs can use later. We will use synthetic data so the model must use tools (not world knowledge).

In [4]:
WEATHER_DB = {
    'AlphaTown': 17.0,
    'BetaVille': 29.0,
    'GammaPort': 24.0,
}


def get_today_date() -> str:
    return date.today().isoformat()


def get_weather(city: str) -> dict[str, Any]:
    if city not in WEATHER_DB:
        raise ValueError(f'Unknown city: {city}')
    return {'city': city, 'temp_c': WEATHER_DB[city]}


def safe_eval(expr: str) -> float:
    """Safely evaluate simple arithmetic expressions for the calculator tool."""
    allowed_ops = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.USub: operator.neg,
    }

    def _eval(node: ast.AST) -> float:
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return float(node.value)
        if isinstance(node, ast.UnaryOp) and type(node.op) in allowed_ops:
            return allowed_ops[type(node.op)](_eval(node.operand))
        if isinstance(node, ast.BinOp) and type(node.op) in allowed_ops:
            return allowed_ops[type(node.op)](_eval(node.left), _eval(node.right))
        raise TypeError(f'Unsupported expression: {ast.dump(node)}')

    tree = ast.parse(expr, mode='eval')
    return round(_eval(tree.body), 4)


def calculator(expression: str) -> float:
    # print(f"[Tool Calling] Calculating expression: {expression}")
    return safe_eval(expression)


def final_answer(answer: str) -> dict[str, str]:
    return {'answer': answer}


TOOLS = {
    'get_today_date': get_today_date,
    'get_weather': get_weather,
    'calculator': calculator,
    'final_answer': final_answer,
}


### Build the Orchestrator Helpers

The orchestrator parses `ACT:` from model text, runs the tool call in Python, and returns an `OBSERVE:` message.

In [5]:
def llm_reply(messages: list[dict[str, str]], model_api_config) -> str:
    """Wrapper for utils.get_completion to always return response text."""
    result = get_completion(messages=messages, model_api_config=model_api_config)
    return result[0] if isinstance(result, tuple) else result


def parse_tool_call(assistant_text: str) -> tuple[str, list[Any], dict[str, Any]]:
    if 'ACT:' not in assistant_text:
        raise ValueError('No ACT section found in assistant response.')

    act_block = assistant_text.split('ACT:', 1)[1].strip()
    call_line = act_block.splitlines()[0].strip()

    call_node = ast.parse(call_line, mode='eval').body
    if not isinstance(call_node, ast.Call) or not isinstance(call_node.func, ast.Name):
        raise ValueError(f'Invalid tool call format: {call_line}')

    tool_name = call_node.func.id
    args = [ast.literal_eval(arg) for arg in call_node.args]
    kwargs = {kw.arg: ast.literal_eval(kw.value) for kw in call_node.keywords}
    return tool_name, args, kwargs


def get_observation_message(assistant_text: str, tools: dict = None) -> str:
    """Parse and execute the tool call in `assistant_text`.

    Args:
        assistant_text: The raw assistant output containing an ACT: block.
        tools: Optional dict of extra (or override) tools merged on top of the
               global TOOLS dict.  Keys in `tools` take precedence over TOOLS.
    """
    _tools = {**TOOLS, **(tools or {})}
    try:
        tool_name, args, kwargs = parse_tool_call(assistant_text)
        if tool_name not in _tools:
            return f'OBSERVE:\nERROR: Unsupported tool `{tool_name}`.'
        output = _tools[tool_name](*args, **kwargs)
        return f'OBSERVE:\n{output}'
    except Exception as exc:
        return f'OBSERVE:\nERROR: {type(exc).__name__}: {exc}'


### 🧑‍🏫 Demo 1: Motivation — "What is today's date?"

LLMs are trained on data with a **knowledge cutoff**. Before Tool Calling (or Function Calling) was developed, they have no access to real-time information, so questions like *"What is today's date?"* expose a fundamental limitation of plain prompting.

Let's compare direct prompting vs a ReAct agent with a `get_today_date` tool.

#### Direct Prompt (No Tools)

In [ ]:
# Direct prompt — no tools
date_question = "What is today's date? Return YYYY-MM-DD only."

direct_response, _ = get_completion(
    user_prompt=date_question,
    model_api_config=model_api_config,
)

print_in_box(date_question, title='User')
print_in_box(direct_response.strip(), title='Assistant (no tools)')

#### ReAct (With Tools)

In [ ]:
date_react_system_prompt = """
You are an agent that solves tasks using the ReAct pattern.

You must always reply in this exact format:
THINK:
<your reasoning about what to do>
ACT:
<one Python-like tool call>

Available tools:
1) get_today_date()
2) final_answer(answer: str)

Rules:
- Use appropriate tools to find the answer.
- After you find the final answer, call final_answer(answer=...) to return your answer.
""".strip()


messages = [
    {'role': 'system', 'content': date_react_system_prompt},
    {'role': 'user', 'content': date_question},
]

max_steps = 4
final_observation = None

for step in range(1, max_steps + 1):
    assistant_text, _ = get_completion(messages=messages, model_api_config=model_api_config)
    observation = get_observation_message(assistant_text)

    messages.append({'role': 'assistant', 'content': assistant_text})
    messages.append({'role': 'user', 'content': observation})

    print_in_box(assistant_text, title=f'Assistant Step {step}')
    print_in_box(observation, title=f'User Step {step}')

    if re.search(r'ACT:\s*final_answer\(', assistant_text):
        final_observation = observation
        break

print('\nDate demo final observation:')
print(final_observation)


**Observation:** 
- The direct prompt lead to a response with an old date based on the model's training data.
- The ReAct agent calls `get_today_date`, receives the real date as an observation, and answers correctly

This is the core value of tool-augmented agents: **the model doesn't need to know everything — it needs to know how to find out.**

### 🧑‍🏫 Demo 2: Full ReAct Loop Demo

Let's build a full ReAct Loop for a more complex task.

In [8]:
def run_react_loop(system_prompt: str, task: str, max_steps: int = 8, verbose: bool = True, tools: dict = None):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': task},
    ]

    final_observation = None

    for step in range(1, max_steps + 1):
        # assistant_text = llm_reply(messages, model_api_config)
        assistant_text, _ = get_completion(messages=messages, model_api_config=model_api_config)
        observation = get_observation_message(assistant_text, tools=tools)

        messages.append({'role': 'assistant', 'content': assistant_text})
        messages.append({'role': 'user', 'content': observation})

        if verbose:
            print_in_box(assistant_text, title=f'Assistant Step {step}')
            print_in_box(observation, title=f'Observe Step {step}')

        if re.search(r'ACT:\s*final_answer\(', assistant_text):
            final_observation = observation
            break

    return messages, final_observation


In [9]:
react_system_prompt = """
You are an agent that solves tasks using the ReAct pattern.

You must always reply in this exact format:
THINK:
<your reasoning about what to do>
ACT:
<one Python-like tool call>

Available tools:
1) get_weather(city: str)
2) calculator(expression: str)
3) final_answer(answer: str)

Rules:
- Use at most one tool call in each response.
- Use tools to get numeric facts and calculations.
- When done, call final_answer(answer=...).
""".strip()

react_task = """
TASK: Find the hottest city among AlphaTown, BetaVille, and GammaPort.
Then compute how many Celsius degrees hotter it is than the coolest city.
""".strip()

_, final_observation = run_react_loop(react_system_prompt, react_task, max_steps=8, verbose=True)
print('\nWeather-task final observation:')
print(final_observation)


╔═══════════════════════════════════════[ Assistant Step 1 ]═══════════════════════════════════════╗
║ THINK:                                                                                           ║
║ I need to find the temperatures of AlphaTown, BetaVille, and GammaPort first to identify the     ║
║ hottest and coolest cities. Then, I will calculate the difference between the hottest and        ║
║ coolest temperatures.                                                                            ║
║ ACT:                                                                                             ║
║ get_weather(city="AlphaTown")                                                                    ║
╚══════════════════════════════════════════════════════════════════════════════════════════════════╝

╔════════════════════════════════════════[ Observe Step 1 ]════════════════════════════════════════╗
║ OBSERVE:                                                                               

### 📝 Exercise 1: Build a ReAct Knowledge Agent with Tools

We will now build the same agent that produced the trace above. We go step by step:

1. Define the data and the tools
2. Write the system prompt
3. Define a task
4. Use the provided simplified orchestrator to run a ReAct loop with the required new tools. 

#### Step 1 — Define the Data and Tools

In [10]:
# A small knowledge base — the agent's 'search engine'
KNOWLEDGE_BASE = {
    'eiffel tower height': 'The Eiffel Tower stands 330 metres tall (including antenna).',
    'eiffel tower location': 'The Eiffel Tower is located in Paris, France.',
    'mount everest height': 'Mount Everest is 8,849 metres above sea level.',
    'speed of light': 'The speed of light in a vacuum is 299,792,458 metres per second.',
    'earth radius': 'The mean radius of the Earth is 6,371 kilometres.',
    'water boiling point': 'Water boils at 100 degrees Celsius (212 degrees Fahrenheit) at sea level.',
    'great wall of china length': 'The Great Wall of China is approximately 21,196 kilometres long.',
    'amazon river length': 'The Amazon River is approximately 6,400 kilometres long.',
    'world population': 'The world population is approximately 8.1 billion people (as of 2024).',
    'python release year': 'Python was first released in 1991 by Guido van Rossum.',
    'moon distance': 'The average distance from Earth to the Moon is 384,400 kilometres.',
    'london population': 'London has a population of approximately 9 million people.',
}


# A simple search tool that matches query words to keys in the KNOWLEDGE_BASE
def search(query: str) -> str:
    query_lower = query.lower()
    query_words = set(query_lower.split())
    best_key, best_score = None, 0
    for key in KNOWLEDGE_BASE:
        key_words = set(key.split())
        score = len(query_words & key_words)
        if score > best_score:
            best_key, best_score = key, score
    if best_key and best_score > 0:
        return KNOWLEDGE_BASE[best_key]
    return 'No relevant information found.'


# Test the search tool
print(search('eiffel tower height metres'))
print(search('amazon river'))

The Eiffel Tower stands 330 metres tall (including antenna).
The Amazon River is approximately 6,400 kilometres long.


#### Step 2 — Write the System Prompt

The system prompt teaches the LLM:
- The exact `THINK / ACT / OBSERVE` format
- What tools are available and how to call them
- The rules to follow in the responses.

In [ ]:
# TODO: Write a system prompt for a ReAct agent to use necessary tools to answer a question about the world knowledge in KNOWLEDGE_BASE.
knowledge_react_system_prompt = """
Your answer here
""".strip()

#### Step 3 — Define a Task

Let's ask the agent to find out the height of the Eiffel Tower in feet (1 metre = 3.28084 ft).

In [12]:
knowledge_react_task = """
TASK: What is the height of the Eiffel Tower in feet? (1 metre = 3.28084 ft)
""".strip()

#### Step 4 — Run a ReAct Loop with Necessary Tools

In [ ]:
# TODO: Run the ReAct loop with your system prompt and the necessary tools, and print the final observation.
_, knowledge_final_observation = "Your code here"

print('\nKnowledge-task final observation:')
print(knowledge_final_observation)

## LLM Feedback Loops
---

A **feedback loop** is a pattern where a system's output is evaluated and the evaluation is fed back as input to improve the next output.

Two common evaluator patterns:

| Pattern | Evaluator | Best For |
|---|---|---|
| **LLM-as-Critic** | A second LLM call evaluates the output and produces natural-language feedback | Open-ended tasks: writing, explanations, design |
| **Programmatic Evaluation** | Code (tests, regex, rules) checks the output deterministically | Code generation, structured output, formatting tasks |

**Key design decisions for feedback loops:**
- **Termination criteria**: all tests pass, a quality score exceeds a threshold, or a maximum number of iterations is reached.
- **Feedback granularity**: vague feedback ("improve the code") is less useful than specific feedback
  ("Test #3 failed: expected `None` for an empty list, got `0`").
- **Context accumulation**: pass only the *current* version + feedback to each iteration,
  not the full history, to avoid context bloat.

### 🧑‍🏫 Demo 3: LLM-as-Critic Pattern

In this pattern, one LLM plays the **generator** (produces a first-draft explanation) and another call
plays the **critic** (evaluates the explanation and suggests specific improvements).
The generator then revises the explanation using the critique.

We'll use the topic *"gradient descent"* — a concept students in AI courses often find hard to grasp.

In [14]:
# Step 1 — Generator: produce a first-draft explanation
topic = 'gradient descent'

generator_prompt = (
    f"Explain '{topic}' in 3-4 sentences for a student who knows basic calculus "
    "but has not studied machine learning before."
)

first_draft, _ = get_completion(
    user_prompt=generator_prompt,
    model_api_config=model_api_config,
)

print_in_box(first_draft.strip(), title='Generator — First Draft')


╔═══════════════════════════════════[ Generator — First Draft ]════════════════════════════════════╗
║ Gradient descent is a method used to find the minimum of a function by moving step-by-step in    ║
║ the direction where the function decreases the fastest. Starting from an initial point, we use   ║
║ the function's derivative (or gradient) to determine this direction and update the point         ║
║ accordingly. By repeating this process, we gradually approach the function's lowest value. This  ║
║ technique is fundamental in machine learning for optimizing models by minimizing errors.         ║
╚══════════════════════════════════════════════════════════════════════════════════════════════════╝


In [15]:
# Step 2 — Critic: evaluate the first draft
critic_prompt = f"""You are an expert educator reviewing a student-facing explanation.

Topic: '{topic}'

Explanation to review:
{first_draft}

Evaluate the explanation on three dimensions and give ONE specific, actionable suggestion per dimension:
1. Clarity — is it easy for a beginner to understand?
2. Accuracy — is it technically correct?
3. Concreteness — does it include an intuitive example or analogy?

Format your response as:
Clarity: <score 1-5> — <one specific suggestion>
Accuracy: <score 1-5> — <one specific suggestion>
Concreteness: <score 1-5> — <one specific suggestion>
Overall: <one sentence summary of the main improvement needed>"""

critique, _ = get_completion(
    user_prompt=critic_prompt,
    model_api_config=model_api_config,
)

print_in_box(critique.strip(), title='Critic — Evaluation')


╔═════════════════════════════════════[ Critic — Evaluation ]══════════════════════════════════════╗
║ Clarity: 4 — Simplify the language by replacing “function's derivative (or gradient)” with “the  ║
║ slope or steepness of the function” to make it more accessible for beginners.                    ║
║ Accuracy: 5 — The explanation is technically correct; no changes needed here.                    ║
║ Concreteness: 2 — Add a simple analogy, such as “imagine walking downhill to reach the lowest    ║
║ point in a valley,” to help students visualize the concept.                                      ║
║ Overall: Including a relatable analogy and simplifying technical terms would make the            ║
║ explanation clearer and more engaging for beginners.                                             ║
╚══════════════════════════════════════════════════════════════════════════════════════════════════╝


In [16]:
# Step 3 — Generator: revise using the critique
revision_prompt = f"""You wrote this explanation of '{topic}':

{first_draft}

An expert reviewer gave the following feedback:
{critique}

Please rewrite the explanation in 3-4 sentences, addressing all the feedback above."""

revised_draft, _ = get_completion(
    user_prompt=revision_prompt,
    model_api_config=model_api_config,
)

print_in_box(first_draft.strip(), title='Before (first draft)')
print_in_box(revised_draft.strip(), title='After (revised with critique)')


╔═════════════════════════════════════[ Before (first draft) ]═════════════════════════════════════╗
║ Gradient descent is a method used to find the minimum of a function by moving step-by-step in    ║
║ the direction where the function decreases the fastest. Starting from an initial point, we use   ║
║ the function's derivative (or gradient) to determine this direction and update the point         ║
║ accordingly. By repeating this process, we gradually approach the function's lowest value. This  ║
║ technique is fundamental in machine learning for optimizing models by minimizing errors.         ║
╚══════════════════════════════════════════════════════════════════════════════════════════════════╝

╔════════════════════════════════[ After (revised with critique) ]═════════════════════════════════╗
║ Gradient descent is a way to find the lowest point of a function by moving step-by-step in the   ║
║ direction where it slopes downward the most. Imagine walking downhill to reach the bott

**Observation:** The critic's structured feedback targets specific weaknesses. The revised explanation
directly addresses each point. This is far more directed than simply re-prompting the generator with
"please improve this".

### 📝 Exercise 2: Build a Coding Agent Step-by-Step

Let's build a proof-of-concept coding agent step-by-step. #Learning by DOING

#### Step 1: Define Helper Functions

In [17]:
def execute_code(code, test_cases):
    """
    Executes Python code and returns the results of test cases.
    Args:
        code: String containing Python code
        test_cases: List of dictionaries with inputs and expected outputs
    Returns:
        Dictionary containing execution results and test outcomes
    """
    results = {"execution_error": None, "test_results": [], "passed": 0, "failed": 0}

    # Create a namespace for execution
    namespace = {}

    # Capture stdout and stderr
    output_buffer = io.StringIO()

    try:
        with redirect_stdout(output_buffer), redirect_stderr(output_buffer):
            exec(code, namespace)

        # Run test cases
        for i, test in enumerate(test_cases):
            inputs = test["inputs"]
            expected = test["expected"]

            # Execute the function with test inputs
            try:
                if isinstance(inputs, dict):
                    actual = namespace["process_data"](**inputs)
                else:
                    actual = namespace["process_data"](*inputs)

                passed = actual == expected

                if passed:
                    results["passed"] += 1
                else:
                    results["failed"] += 1

                results["test_results"].append(
                    {
                        "test_id": i + 1,
                        "inputs": inputs,
                        "expected": expected,
                        "actual": actual,
                        "passed": passed,
                    }
                )
            except Exception as e:
                # If the error is the expected type, mark as passed
                passed = isinstance(expected, type) and isinstance(e, expected)
                results["test_results"].append(
                    {
                        "test_id": i + 1,
                        "inputs": inputs,
                        "expected": expected,
                        "error": str(e),
                        "passed": passed,
                    }
                )
                if passed:
                    results["passed"] += 1
                else:
                    results["failed"] += 1

    except Exception as e:
        results["execution_error"] = {
            "error_type": type(e).__name__,
            "error_message": str(e),
            "traceback": traceback.format_exc(),
        }

    results["stdout"] = output_buffer.getvalue()
    return results


# Function to format test results as feedback for the model
def format_feedback(results):
    """
    Formats test results into a clear feedback string for the model.
    Args:
        results: Dictionary containing execution results
    Returns:
        Formatted feedback string
    """
    feedback = []

    if results["execution_error"]:
        feedback.append(
            f"ERROR: Code execution failed with {results['execution_error']['error_type']}"
        )
        feedback.append(f"Message: {results['execution_error']['error_message']}")
        feedback.append("Traceback:")
        feedback.append(results["execution_error"]["traceback"])
        feedback.append("\nPlease fix the syntax or runtime errors in the code.")
        return "\n".join(feedback)

    feedback.append(
        f"Test Results: {results['passed']} passed, {results['failed']} failed"
    )

    if results["stdout"]:
        feedback.append(f"\nStandard output:\n{results['stdout']}")

    if results["failed"] > 0:
        feedback.append("\nFailed Test Cases:")
        for test in results["test_results"]:
            if not test.get("passed"):
                feedback.append(f"\nTest #{test['test_id']}:")
                feedback.append(f"  Inputs: {test['inputs']}")
                feedback.append(f"  Expected: {test['expected']}")
                if "actual" in test:
                    feedback.append(f"  Actual: {test['actual']}")
                if "error" in test:
                    feedback.append(f"  Error: {test['error']}")

    return "\n".join(feedback)

#### Step 2: Define Task and Test Cases

We will create a Python function called `process_data` that analyses numerical data with the following (possibly incomplete) set of requirements:

1. The function should accept a list of numbers and an optional parameter 'mode' that can be 'sum' or 'average' (default should be 'average').
2. If mode is 'sum', return the sum of all numbers.
3. If mode is 'average', return the average (mean) of all numbers.

Example:
```python
process_data([1, 2, 3, 4, 5], mode='average')  # Should return 3.0
process_data([1, 2, 'a', 3], mode='sum')  # Should return 6
```

In [18]:
# Write out a task description for the LLM
task_description = """
We will create a Python function called `process_data` that analyzes numerical data with the following (possibly incomplete) set of requirements:

1. The function should accept a list of numbers and an optional parameter 'mode' that can be 'sum' or 'average' (default should be 'average').
2. If mode is 'sum', return the sum of all numbers.
3. If mode is 'average', return the average (mean) of all numbers.

Example:
```python
process_data([1, 2, 3, 4, 5], mode='average')  # Should return 3.0
process_data([1, 2, 'a', 3], mode='sum')  # Should return 6
```
"""

In [19]:
# Write out test cases to test the LLM's output
# TODO: You can add more test cases following the examples below.
test_cases = [
    {"inputs": ([1, 2, 3, 4, 5], "sum"), "expected": 15},
    {"inputs": ([1, 2, 3, 4, 5], "average"), "expected": 3.0},
    {"inputs": ([11, 12, 13, 14, 15], "sum"), "expected": 65},
    {"inputs": ([11, 12, 13, 14, 15], "average"), "expected": 13.0},
    {"inputs": ([1.1, 2.2, 3.3, 4.4, 5.5], "sum"), "expected": 16.5},
    {"inputs": ([1.1, 2.2, 3.3, 4.4, 5.5], "average"), "expected": 3.3},
    {"inputs": ([-1, -2, -3, -4, -5], "sum"), "expected": -15},
    {"inputs": ([-1, -2, -3, -4, -5], "average"), "expected": -3.0},
]

#### Step 3: Initial Generation

Let's start with a basic prompt to generate an initial solution to our problem.

In [ ]:
# Basic prompt for initial code generation
# TODO: Complete the prompt
initial_prompt = f"""
You are an expert Python developer. Please write a Python function based on the following requirements:

<Your code here>

Write only the function surrounded by ```python and ``` without any additional explanations or examples.

Example:

```python
def function_name(arguments):
    # function body
```
"""

# TODO: Get initial completion
messages = [{"role": "user", "content": initial_prompt}]
initial_response, _ = "Your code here"

def extract_code(code):
    lines = code.split("\n")
    start = lines.index("```python") + 1
    end = lines.index("```", start)
    return "\n".join(lines[start:end])

# TODO: Extract the code from the LLM response and print it out.
initial_code = "Your code here"

print("Initial Generated Code:")
print(initial_code)

# TODO: Execute and test the initial code
initial_results = "Your code here"

# TODO: Format the test results into feedback for the LLM and print it out.
initial_feedback = "Your code here"

print("\nTest Results:")
print(initial_feedback)

#### Step 4: Expand the Test Cases

The first version of your generated code worked perfectly, Now, your product manager asked you to:
1) support a new mode, "median"
2) ignore non-numeric values
3) handle empty lists, returning None

So, following test-driven development practices, you update your tests:

In [21]:
# These are the new test cases. No updates needed.
test_cases = [
    {"inputs": ([1, 2, 3, 4, 5], "sum"), "expected": 15},
    {"inputs": ([1, 2, 3, 4, 5], "average"), "expected": 3.0},
    {"inputs": ([11, 12, 13, 14, 15], "sum"), "expected": 65},
    {"inputs": ([11, 12, 13, 14, 15], "average"), "expected": 13.0},
    {"inputs": ([], "sum"), "expected": None},
    {"inputs": ([1, 3, 4], "median"), "expected": 3},
    {"inputs": ([1, 2, 3, 5], "median"), "expected": 2.5},
    {"inputs": ([1, 2, "a", 3], "sum"), "expected": 6},
    {"inputs": ([1, 2, None, 3, "b", 4], "average"), "expected": 2.5},
    {"inputs": ([10], "median"), "expected": 10},
    {"inputs": ([], "median"), "expected": None},
    {"inputs": ([1, 2, 3, 4, 5], "invalid_mode"), "expected": ValueError},
]

Let's re-test the code with the updated test cases.

In [22]:
# Re-test the code
# No updates are needed in this cell
print("Initial Generated Code:")
print(initial_code)

# Execute and test the initial code
initial_results = execute_code(initial_code, test_cases)
initial_feedback = format_feedback(initial_results)

print("\nTest Results:")
print(initial_feedback)

Initial Generated Code:
def process_data(numbers, mode='average'):
    filtered_numbers = [n for n in numbers if isinstance(n, (int, float))]
    if not filtered_numbers:
        return 0
    if mode == 'sum':
        return sum(filtered_numbers)
    elif mode == 'average':
        return sum(filtered_numbers) / len(filtered_numbers)
    else:
        raise ValueError("mode must be 'sum' or 'average'")

Test Results:
Test Results: 7 passed, 5 failed

Failed Test Cases:

Test #5:
  Inputs: ([], 'sum')
  Expected: None
  Actual: 0

Test #6:
  Inputs: ([1, 3, 4], 'median')
  Expected: 3
  Error: mode must be 'sum' or 'average'

Test #7:
  Inputs: ([1, 2, 3, 5], 'median')
  Expected: 2.5
  Error: mode must be 'sum' or 'average'

Test #10:
  Inputs: ([10], 'median')
  Expected: 10
  Error: mode must be 'sum' or 'average'

Test #11:
  Inputs: ([], 'median')
  Expected: None
  Actual: 0


#### Step 5: First Iteration with Feedback

Now, let's feed the test results back to the model and ask for an improved version.

In [ ]:
# Prompt with feedback for first iteration
# TODO: Complete the prompt to include the initial code and the feedback from testing that code.
feedback_prompt = f"""
You are an expert Python developer. You wrote a function based on these requirements:

<Your code here>

Here is your current implementation:
```python
<Your code here>
```
I've tested your code and here are the results:
<Your code here>
Please improve your code to fix any issues and make sure it passes all test cases.
Write only the improved function without any explanation.
"""

messages = [{"role": "user", "content": feedback_prompt}]

# TODO: Get improved code
improved_response, _ = "Your code here"

# TODO: Extract the improved code
improved_code = "Your code here"

print("\nImproved Code:")
print(improved_code)

# TODO: Execute and test the improved code
improved_results = "Your code here"
improved_feedback = "Your code here"
print("\nTest Results for Improved Code:")
print(improved_feedback)


#### Step 6: Create Feedback Loop

We may want to give the LLM more than one chance to generate the correct code. We may even want to introduce test cases gradually, so that it has the opportunity to fix errors one at a time.

Let's develop a loop that will start from scratch and run the loop a maximum number of times or until the code is correct.

In [ ]:
# Write a code-creation LLM loop
from pprint import pprint

iterations = []


# TODO: Get initial completion and extract code
messages = [{"role": "user", "content": initial_prompt}]
initial_response, _ = "Your code here"
initial_code = "Your code here"

# TODO: Execute and test the initial code
initial_results = "Your code here"
initial_feedback = "Your code here"


# Store the initial iteration
iterations.append(
    {
        "iteration": 0,
        "code": initial_code,
        "test_results": {
            "passed": initial_results["passed"],
            "failed": initial_results["failed"],
        },
    }
)

pprint(iterations[-1]["test_results"])

current_code = initial_code
current_feedback = initial_feedback

# TODO: Complete the loop to improve the code based on feedback
for i in range(3):
    # Check if all tests passed
    if iterations[-1]["test_results"]["failed"] == 0:
        print("\nSuccess! All tests passed.")
        break

    "Your code here"
    
    pprint(iterations[-1]["test_results"])

    current_code = improved_code
    current_feedback = improved_feedback


### Structured Data in Production Agentic Systems: 

While the examples in this notebook use dictionaries and text-based prompts for messages to AI agents, real-world production agentic systems rely heavily on JSON and Pydantic to ensure robustness and type safety. 
- JSON provides a language-agnostic format for serializing agent inputs, outputs, and tool responses
- Pydantic offers Python-based validation and serialization that guarantees data conforms to expected schemas at runtime. Using Pydantic models to define agent tool parameters, API responses, and message structures prevents silent data errors, provides automatic documentation through schema generation, and enables seamless integration with modern AI platforms and APIs. 
- As you scale your agentic applications, adopting structured data patterns with JSON and Pydantic becomes essential for maintainability, error detection, and interoperability across distributed systems.


## Summary
---

| Technique | Core Idea | Key Mechanism | Typical Use Case |
|---|---|---|---|
| **Chain-of-Thought** | Add explicit reasoning steps | Single prompt with intermediate reasoning | Multi-step reasoning without external info |
| **ReAct** | Interleave reasoning with tool use | Thought → Action → Observation loop (multi-turn) | Tasks needing real-time data or computation |
| **LLM-as-Critic** | A second LLM evaluates and critiques the output | Generator → Critic → Revised Generator (iterative) | Open-ended writing, explanations, design |
| **Programmatic Feedback Loop** | Code checks the output deterministically | Generator → Test Runner → Feedback → Generator | Code generation, structured output, format validation |

**When to choose which:**
- Use **ReAct** when the task requires up-to-date information or computation the LLM cannot do reliably alone.
- Use **LLM-as-Critic** when the quality of an open-ended response is hard to measure programmatically.
- Use **Programmatic Feedback** when correctness can be verified by code — tests are cheaper and more precise
  than a second LLM call.

